In [ ]:
# 1. Problem Definition & Objective:

#     - a. Selected Project Track:
# This project falls under the Content Recommendation System track, combining traditional machine learning with modern semantic search techniques.

#     - b. Clear Problem Statement:
# Choosing the right car is difficult due to the large number of available options and the complexity of user preferences. Traditional filters based on price or brand do not capture semantic intent such as “family-friendly”, “sporty” or “fuel efficient”.

# The goal of this project is to build an Intelligent Car Recommendation System that understands natural language queries and recommends suitable cars based on semantic similarity and relevance.

#      - c. Real-World Relevance and Motivation:
# This system can be applied in:
# - Online car marketplaces.
# - Automotive recommendation platforms.
# - Sales decision support tools.

# It improves user experience by allowing free-text queries instead of rigid filters, making car selection more intuitive and personalized.

In [ ]:
# 🚗 Intelligent Car Recommendation System
## Using ML, NLP, and LLM Techniques

# **Components:**
# - Flan-T5 for natural language understanding.
# - Sentence Transformers for semantic search.
# - Traditional ML (Scikit-learn, TF-IDF, KNN).
# - Rule-based filtering.

# **Dataset:** Kaggle Cars Dataset 2025

In [ ]:
# 1. Installation & Imports

In [ ]:
# Install required packages
%pip install transformers sentence-transformers scikit-learn pandas numpy torch kaggle -q

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# NLP & Transformers
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Utilities
from typing import List, Dict
import re

In [ ]:
# 2. Data Understanding & Preparation:

# a. Dataset Source:
# The dataset used in this project is a structured CSV dataset which is extracted from kaggle (https://www.kaggle.com/datasets/abdulmalik1518/cars-datasets-2025/data) containing car specifications such as:
# - Company name
# - Car name
# - Price
# - Fuel type
# - Engine details
# - Horsepower
# - Seats
# - Speed
# - Description text

# b. Data Loading and Exploration:
# The dataset is loaded using Pandas, and initial exploration includes:
# - Checking column names.
# - Inspecting sample rows.
# - Understanding data types and value distributions.
# Basic exploratory steps help ensure the dataset aligns with the recommendation goals.

# c. Cleaning, Preprocessing, and Feature Engineering:
# Key preprocessing steps include:
# - Selecting relevant text columns (e.g., car descriptions).
# - Converting text to lowercase.
# - Removing stop words (for TF-IDF).
# - Combining multiple attributes into descriptive text when required.

# d. Handling Missing Values or Noise:
# - Rows with critical missing information are removed or ignored.
# - Non-critical missing values are handled gracefully during recommendation.
# - Text-based models are naturally robust to minor noise.

In [ ]:
# 2. Load and Explore Dataset

In [ ]:
import pandas as pd
# Load the dataset
# Make sure to download from Kaggle and place in your working directory
# Or use Kaggle API: !kaggle datasets download -d abdulmalik1518/cars-datasets-2025

df = pd.read_csv('cars_dataset_2025.csv', encoding='latin1')  # Adjust filename as needed and try 'latin1' encoding

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Explore dataset structure
print("Column Names:")
print(df.columns.tolist())

print("\nDataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# 3. Model / System Design:

# a. AI Techniques Used:
# This project uses a hybrid AI approach:
# - TF-IDF for keyword-based similarity.
# - Sentence Transformers for semantic similarity.
# - Large Language Models (Flan-T5) for reasoning and explanation.
# - A recommendation system architecture.

# b. Architecture / Pipeline Explanation:
# The pipeline follows these steps:
# 1. User enters a natural language query.
# 2. Query is vectorized using TF-IDF and sentence embeddings.
# 3. Similarity is computed against car descriptions.
# 4. Scores are combined to rank cars.
# 5. Top results are returned to the user via a web interface.

# c. Justification of Design Choices:
# - TF-IDF provides fast lexical matching.
# - Sentence embeddings capture semantic meaning.
# - Hybrid scoring improves robustness.
# - Gradio enables rapid deployment and user interaction.

In [ ]:
# 3. Data Preprocessing

In [ ]:
# Handle missing values
df = df.dropna(subset=['Company Names', 'Cars Names'])
df = df.fillna('')

def create_car_description(row):
    desc_parts = []

    if 'Company Names' in row and row['Company Names']:
        desc_parts.append(str(row['Company Names']))
    if 'Cars Names' in row and row['Cars Names']:
        desc_parts.append(str(row['Cars Names']))
    if 'Fuel Types' in row and row['Fuel Types']:
        desc_parts.append(str(row['Fuel Types']))
    if 'Engines' in row and row['Engines']:
        desc_parts.append(str(row['Engines']))
    if 'CC/Battery Capacity' in row and row['CC/Battery Capacity']:
        desc_parts.append(str(row['CC/Battery Capacity']))
    if 'HorsePower' in row and row['HorsePower']:
        desc_parts.append(str(row['HorsePower']))
    if 'Cars Prices' in row and row['Cars Prices']:
        desc_parts.append(str(row['Cars Prices']))
    if 'Seats' in row and row['Seats']:
        desc_parts.append(str(row['Seats']))
    if 'Torque' in row and row['Torque']:
        desc_parts.append(str(row['Torque']))

    return ' '.join(desc_parts)

df['description'] = df.apply(create_car_description, axis=1)

# Clean price column
def clean_price(price_str):
    if pd.isna(price_str) or price_str == '':
        return 0
    price_str = str(price_str).replace('$', '').replace(',', '').strip()
    try:
        return float(price_str)
    except:
        return 0

df['Price_Numeric'] = df['Cars Prices'].apply(clean_price)

print("Sample descriptions:")
print(df['description'].head(3))

In [ ]:
# 4. Core Implementation:

# a. Model Training / Inference Logic:
# - TF-IDF vectors are computed from car descriptions.
# - SentenceTransformer generates dense semantic embeddings.
# - Cosine similarity is used to measure relevance.
# - No supervised training is required, the system is inference-based.

# b. Prompt Engineering (LLM-Based Components):
# For LLM usage:
# - Prompts are designed to explain why a car matches a query.
# - The model generates human-readable explanations.
# - Prompts are concise and context-aware.

# c. Recommendation Pipeline:
# 1. Accept user query.
# 2. Generate vector representations.
# 3. Compute similarity scores.
# 4. Rank cars by relevance.
# 5. Display results with scores and attributes.

# d. Code Execution Guarantee:
# The notebook is structured to:
# - Run top-to-bottom without errors.
# - Load all dependencies before execution.
# - Initialize models only once.

In [ ]:
# 4. Initialize Models

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
# Load Sentence Transformer
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# Load Flan-T5
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-small')
llm_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small')

print("Models loaded successfully")

In [ ]:
# 5. Evaluation & Analysis:

# a. Metrics Used:
# Since this is an unsupervised recommendation system, evaluation is primarily qualitative:
# - Relevance of recommendations.
# - Semantic alignment with user intent.
# - Consistency across similar queries.

# b. Sample Outputs:
# Example queries:
# - “Affordable sports car with good performance”.
# - “Electric SUV for family use”.
# - “Luxury car with high horsepower”.
# The system returns ranked car recommendations with similarity scores.

# c. Performance Analysis and Limitations:
# Strengths:
# - Understands natural language.
# - Flexible and user-friendly.
# - Fast inference.

# Limitations:
# - Depends on dataset quality.
# - No explicit user feedback loop.
# - Limited personalization without user history.

In [ ]:
# 5. Method 1: TF-IDF + Cosine Similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['description'])

def tfidf_recommend(query, top_n=5):
    query_vec = tfidf_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[-top_n:][::-1]

    results = df.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    return results

In [ ]:
# 6. Ethical Considerations & Responsible AI:

# a. Bias and Fairness Considerations:
# - Recommendations may reflect biases present in the dataset.
# - Popular brands may appear more frequently.
# - No demographic or personal user data is used.

# b. Dataset Limitations:
# - Dataset may not cover all car models.
# - Prices and specifications may become outdated.
# - Images and descriptions may vary in quality.

# c. Responsible Use of AI Tools:
# - System is designed for informational purposes only.
# - Recommendations should not replace professional advice.
# - Transparency is maintained in how results are generated.

In [ ]:
# 6. Method 2: K-Nearest Neighbors (KNN)

In [ ]:
numerical_cols = ['Year', 'Price', 'Mileage', 'Engine_Size']
available_numerical = [col for col in numerical_cols if col in df.columns]

if available_numerical:
    for col in available_numerical:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df[available_numerical])

    knn_model = NearestNeighbors(n_neighbors=10)
    knn_model.fit(scaled_features)

In [ ]:
def knn_recommend(query_features, top_n=5):
    query_vec = np.array([[query_features.get(col, df[col].median())
                           for col in available_numerical]])
    query_scaled = scaler.transform(query_vec)
    distances, indices = knn_model.kneighbors(query_scaled, n_neighbors=top_n)

    results = df.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    return results

In [ ]:
# 7. Conclusion & Future Scope:

# a. Summary of Results:
# This project successfully demonstrates an AI-powered car recommendation system that:
# - Understands natural language queries.
# - Combines multiple AI techniques.
# - Provides meaningful and relevant recommendations.
# - Offers a modern, interactive user interface.

# b. Future Improvements and Extensions:
# Possible enhancements include:
# - User profile based personalization.
# - Feedback driven learning.
# - Real time data integration via APIs.
# - Image based recommendation.
# - Deployment as a full web application.
# The system provides a strong foundation for intelligent recommendation platforms.

In [ ]:
# 7. Sentence Transformers (Semantic Search)

In [ ]:
car_embeddings = sentence_model.encode(df['description'].tolist(), show_progress_bar=True)

def semantic_recommend(query, top_n=5):
    query_embedding = sentence_model.encode([query])
    similarities = cosine_similarity(query_embedding, car_embeddings).flatten()
    top_indices = similarities.argsort()[-top_n:][::-1]

    results = df.iloc[top_indices].copy()
    results['semantic_score'] = similarities[top_indices]
    return results

In [ ]:
# 8. Rule-Based Filtering

In [ ]:
def apply_filters(df_subset, filters):
    filtered = df_subset.copy()

    if 'max_price' in filters and 'Price_Numeric' in filtered.columns:
        filtered = filtered[filtered['Price_Numeric'] <= filters['max_price']]
    if 'min_price' in filters and 'Price_Numeric' in filtered.columns:
        filtered = filtered[filtered['Price_Numeric'] >= filters['min_price']]

    if 'min_year' in filters and 'Year' in filtered.columns:
        filtered = filtered[filtered['Year'] >= filters['min_year']]

    if 'fuel_type' in filters and 'Fuel_Type' in filtered.columns:
        filtered = filtered[filtered['Fuel_Type'].str.contains(
            filters['fuel_type'], case=False, na=False)]

    return filtered

In [ ]:
# 9. Hybrid Recommendation System

In [ ]:
def hybrid_recommend(query, filters=None, top_n=5):
    tfidf_res = apply_filters(tfidf_recommend(query, top_n*2), filters or {})
    semantic_res = apply_filters(semantic_recommend(query, top_n*2), filters or {})

    combined = {}
    for idx in set(tfidf_res.index) | set(semantic_res.index):
        combined[idx] = (
            tfidf_res.get('similarity_score', {}).get(idx, 0) * 0.4 +
            semantic_res.get('semantic_score', {}).get(idx, 0) * 0.6
        )

    top = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_n]
    results = df.loc[[i for i, _ in top]].copy()
    results['recommendation_score'] = [s for _, s in top]
    return results

In [ ]:
# Test Query
query = "cheap sports car with good performance"
results = hybrid_recommend(query, top_n=5)

# Check what we got back
if isinstance(results, tuple):
    if len(results) == 3:
        results, understanding, filters = results
    else:
        results = results[0]  # Just get the first item

print("\n🎯 Top Recommendations:")
print(results[['Company Names', 'Cars Names', 'Cars Prices', 'Fuel Types']])

In [ ]:
# 10. User-Friendly Interface

def car_recommendation_system():
    """
    Interactive car recommendation system
    """
    print("="*70)
    print("🚗 WELCOME TO INTELLIGENT CAR RECOMMENDATION SYSTEM")
    print("="*70)
    print("\nI can help you find the perfect car based on your preferences!")
    print("\nExamples of queries you can ask:")
    print("  - 'affordable sports car with good performance'")
    print("  - 'electric SUV under $50000'")
    print("  - 'family sedan with good fuel economy'")
    print("  - 'luxury car with high horsepower'")
    print("\n" + "-"*70)

    while True:
        print("\n")
        query = input("🔍 What kind of car are you looking for? (or type 'quit' to exit): ").strip()

        if query.lower() in ['quit', 'exit', 'q', '']:
            print("\n✨ Thank you for using the Car Recommendation System! Goodbye!")
            break

        print(f"\n⏳ Searching for: '{query}'...")
        print("Please wait...\n")

        try:
            # Get ALL recommendations (change top_n to large number)
            results = hybrid_recommend(query, top_n=50)  # Get top 50 matches

            if len(results) == 0:
                print("❌ No cars found matching your criteria. Try a different query!")
                continue

            # Display results
            print("="*70)
            print(f"🎯 FOUND {len(results)} CARS MATCHING YOUR CRITERIA:")
            print("="*70)

            for i, (idx, row) in enumerate(results.iterrows(), 1):
                print(f"\n{i}. {row['Company Names']} {row['Cars Names']}")
                print(f"   💰 Price: {row['Cars Prices']}")
                print(f"   ⛽ Fuel Type: {row['Fuel Types']}")
                print(f"   🔋 Engine: {row['Engines']}")
                print(f"   ⚡ Horsepower: {row['HorsePower']}")
                print(f"   💺 Seats: {row['Seats']}")
                print(f"   📊 Match Score: {row['recommendation_score']:.2%}")

            print("\n" + "-"*70)
            print(f"Total Results: {len(results)} cars")

            # Ask if user wants to see more details
            more = input("\n📋 Would you like to see detailed info for a specific car? (enter number or 'no'): ").strip()
            if more.isdigit() and 1 <= int(more) <= len(results):
                car_num = int(more) - 1
                row = results.iloc[car_num]
                print(f"\n{'='*70}")
                print(f"DETAILED INFO - {row['Company Names']} {row['Cars Names']}")
                print(f"{'='*70}")
                print(f"Company: {row['Company Names']}")
                print(f"Model: {row['Cars Names']}")
                print(f"Price: {row['Cars Prices']}")
                print(f"Engine: {row['Engines']}")
                print(f"CC/Battery: {row['CC/Battery Capacity']}")
                print(f"Horsepower: {row['HorsePower']}")
                print(f"Top Speed: {row['Total Speed']}")
                print(f"0-100 km/h: {row['Performance(0 - 100 )KM/H']}")
                print(f"Fuel Type: {row['Fuel Types']}")
                print(f"Seats: {row['Seats']}")
                print(f"Torque: {row['Torque']}")
                print(f"Match Score: {row['recommendation_score']:.2%}")

        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            print("Please try again with a different query.")

        print("\n" + "="*70)

# Run the system
car_recommendation_system()

In [ ]:
# 11. Beautiful Web UI using Gradio

%pip install gradio -q
import gradio as gr

# ---------- LUCIDE SVG ICONS (INLINE) ----------
ICON_COLOR = "#60A5FA"
ICON_SIZE = 18

ICON_SEARCH = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<circle cx="11" cy="11" r="8"></circle>
<line x1="21" y1="21" x2="16.65" y2="16.65"></line>
</svg>
"""

ICON_PRICE = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<path d="M21 12V7a2 2 0 0 0-2-2H5"/>
<path d="M3 7v10a2 2 0 0 0 2 2h14"/>
<path d="M16 12h.01"/>
</svg>
"""

ICON_FUEL = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<path d="M3 22V4a2 2 0 0 1 2-2h10a2 2 0 0 1 2 2v18"/>
<path d="M14 10h4l3 3v7a2 2 0 0 1-2 2h-1"/>
</svg>
"""

ICON_ENGINE = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<rect x="3" y="7" width="18" height="10" rx="2"/>
<path d="M7 7V5M17 7V5M7 17v2M17 17v2"/>
</svg>
"""

ICON_POWER = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<path d="M13 2L3 14h7v8l10-12h-7z"/>
</svg>
"""

ICON_SPEED = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<circle cx="12" cy="12" r="10"/>
<path d="M12 12l4-4"/>
</svg>
"""

ICON_SEATS = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<path d="M6 3v12"/>
<path d="M18 3v12"/>
<path d="M4 15h16"/>
</svg>
"""

ICON_TORQUE = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<circle cx="12" cy="12" r="3"/>
<path d="M19.4 15a7.97 7.97 0 0 0 .6-3"/>
</svg>
"""

ICON_CC = f"""
<svg width="{ICON_SIZE}" height="{ICON_SIZE}" viewBox="0 0 24 24" fill="none"
stroke="{ICON_COLOR}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
<rect x="3" y="4" width="18" height="16" rx="2"/>
</svg>
"""

# ---------- SEARCH FUNCTION ----------
def search_cars(query, num_results):
    try:
        results = hybrid_recommend(query, top_n=int(num_results))

        if len(results) == 0:
            return "No cars found. Try a different query."

        output = f"""
        <div style="font-family: Inter, Arial, sans-serif; color: #E5E7EB;">
            <h2 style="font-size:1.6rem; font-weight:600; color:#93C5FD;">
                Found {len(results)} Cars Matching Your Criteria
            </h2>
            <hr style="border-color:#334155;">
        """

        for i, (_, row) in enumerate(results.iterrows(), 1):
            score = row['recommendation_score']
            score_color = "#22C55E" if score >= 0.6 else "#F59E0B"

            output += f"""
            <div style="
                background: linear-gradient(145deg,#0F172A,#020617);
                border:1px solid #1E293B;
                border-radius:16px;
                padding:18px;
                margin-bottom:16px;
            ">
                <h3 style="color:#F8FAFC;">{i}. {row['Company Names']} {row['Cars Names']}</h3>

                <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:12px;">
                    <div>{ICON_PRICE} <b>Price:</b> {row['Cars Prices']}</div>
                    <div>{ICON_FUEL} <b>Fuel:</b> {row['Fuel Types']}</div>
                    <div>{ICON_ENGINE} <b>Engine:</b> {row['Engines']}</div>
                    <div>{ICON_POWER} <b>Power:</b> {row['HorsePower']}</div>
                    <div>{ICON_SPEED} <b>Top Speed:</b> {row['Total Speed']}</div>
                    <div>{ICON_SEATS} <b>Seats:</b> {row['Seats']}</div>
                    <div>{ICON_TORQUE} <b>Torque:</b> {row['Torque']}</div>
                    <div>{ICON_CC} <b>CC:</b> {row['CC/Battery Capacity']}</div>
                </div>

                <div style="
                    margin-top:12px;
                    background:{score_color};
                    color:#020617;
                    padding:8px;
                    border-radius:999px;
                    width:fit-content;
                    font-weight:600;
                ">
                    {score:.1%} match
                </div>
            </div>
            """

        return output + "</div>"

    except Exception as e:
        return str(e)

# ---------- UI ----------
custom_css = """
body { 
    background: radial-gradient(circle at top, #180501, #020617); 
}
footer { 
    display:none; 
}
"""

with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    title="AI Car Recommendation System"
) as demo:

    # ---------- HEADER ----------
    gr.Markdown("""
    <div style="text-align:center; margin-bottom: 30px;">
        <h1 style="font-size: 2.4rem; font-weight: 700; color: #F8FAFC;">
            Car Recommendation System
        </h1>
        <p style="color:#94A3B8; font-size:1rem;">
            Find your perfect car using AI-powered recommendations<br>
            Project by Suganth S [iitrpr_ai_25010888]
        </p>
    </div>
    """)

    # ---------- SEARCH ROW ----------
    with gr.Row():
        with gr.Column(scale=4):
            query_input = gr.Textbox(
                label="What kind of car are you looking for?",
                placeholder="e.g. affordable sports coupe, electric SUV, luxury sedan",
                lines=2
            )

        with gr.Column(scale=1):
            num_results = gr.Slider(
                minimum=5,
                maximum=50,
                value=10,
                step=5,
                label="Results"
            )

    search_btn = gr.Button("Search Cars")

    # ---------- OUTPUT ----------
    output = gr.HTML()

    # ---------- EXAMPLES (BELOW RESULTS) ----------
    gr.Markdown("""
    <h3 style="margin-top:30px; color:#CBD5F5;">
        Try Example Searches
    </h3>
    """)

    gr.Examples(
        examples=[
            ["top 5 coupe cars", 5],
            ["electric SUV under $50,000", 10],
            ["family sedan with good mileage", 10],
            ["luxury car with high horsepower", 10],
            ["sports car like Ferrari or Lamborghini", 10],
        ],
        inputs=[query_input, num_results]
    )

    # ---------- ACTION ----------
    search_btn.click(
        fn=search_cars,
        inputs=[query_input, num_results],
        outputs=output
    )

demo.launch(share=True, debug=True)

In [ ]:
print("✓ All libraries imported successfully!")
print(f"✓ Dataset loaded: {df.shape[0]} cars")
print("✓ Models ready!")